# Selección y Transformación de Variables

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv('data/income_data.csv')

# Limpieza igual que en el EDA
df = df.drop_duplicates()
df = df[df['occupation'] != '?']
df = df[df['workclass'] != '?']
df = df[df['native-country'] != '?']

print(f"Dataset cargado: {df.shape}")
df.head()

Dataset cargado: (45175, 15)


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K


## 1. Selección de Variables

CDespues del EDA, tomamos las siguientes decisiones:

- fnlwgt: Variable de peso muestral administrativo sin valor predictivo real. La correlación con el target era prácticamente 0, así que la eliminamos.
- education: Contiene la misma información que educational-num pero en texto. Como ya tenemos la versión numérica ordenada, esta columna es redundante y la descartamos.

El resto de variables se mantienen para el modelado.

In [3]:
# Eliminamos variables menciaonadas antes
df = df.drop(columns=['fnlwgt', 'education'])

# Separamos features y target
X = df.drop(columns=['income'])
y = df['income']

print("Variables seleccionadas:", X.columns.tolist())
print("Dimensiones X:", X.shape)

Variables seleccionadas: ['age', 'workclass', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
Dimensiones X: (45175, 12)


## 2. Codificación de Variables Categóricas



Label Encoding

Hemos usado Label Encoding para las variables binarias (como el género) y para las que tienen muy pocas categorías.

Variables que vamos a usar:
- gender: binaria (Male / Female)
- marital-status: 7 categorías, sin orden pero pocas valores
- relationship: 6 categorias

In [ ]:
le = LabelEncoder()

cols_label = ['gender', 'marital-status', 'relationship']
# Optimizado con IA: bucle para aplicar LabelEncoder a la lista de columnas de forma limpia
for col in cols_label:
    X[col] = le.fit_transform(X[col])
    print(f"{col}: {sorted(X[col].unique())}")

gender: [np.int64(0), np.int64(1)]
marital-status: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
relationship: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


One-Hot Encoding

Aplicamos One-Hot Encoding a variables nominales con más categorías donde no existe un orden lógico entre ellas. De esta forma evitamos que el modelo interprete relaciones de orden inexistentes.

Variables seleccionadas:
- workclass: 8 categorías (tipo de empleador)
- occupation: 14 ocupaciones laborales distintas
- race: 5 categorías

In [ ]:
cols_ohe = ['workclass', 'occupation', 'race']

# Optimizado con IA: uso del drop_first=True para evitar la multicolinealidad
X = pd.get_dummies(X, columns=cols_ohe, drop_first=True)

print(f"Dimensiones tras One-Hot Encoding: {X.shape}")

Dimensiones tras One-Hot Encoding: (45175, 32)


Agrupación de native-country

Esta variable tiene 41 categorías, pero más del 90% de los registros son de United-States. COn One-Hot Encoding se harían muchas columnas con muy poca información, así que optamos por agrupar en dos valores: United-States y Other.

In [ ]:
X['native-country'] = X['native-country'].apply(
    lambda x: 'United-States' if x == 'United-States' else 'Other'
)
X['native-country'] = le.fit_transform(X['native-country'])

print("native-country valores:", X['native-country'].value_counts().to_dict())
#Con ayuda de IA para llegar a este resultado (lambda para agrupar rápidamente las categorías minoritarias)

native-country valores: {1: 41256, 0: 3919}


## 3. Codificación del Target

Convertimos income a binario: 0 para <=50K y 1 para >50K.

In [7]:
y = y.apply(lambda x: 1 if '>50K' in str(x) else 0)

print("Distribución del target:")
print(y.value_counts())
print(f"\nPorcentaje clase positiva (>50K): {y.mean():.2%}")

Distribución del target:
income
0    33973
1    11202
Name: count, dtype: int64

Porcentaje clase positiva (>50K): 24.80%


## 4. Escalado de Variables Numéricas

Aunque Random Forest y LightGBM no son sensibles al escalado, aplicamos StandardScaler a las variables numéricas continuas para dejar el pipeline preparado y bien documentado.

In [8]:
cols_numericas = ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']

scaler = StandardScaler()
X[cols_numericas] = scaler.fit_transform(X[cols_numericas])

print("Escalado aplicado a:", cols_numericas)
X[cols_numericas].describe().round(2)

Escalado aplicado a: ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']


,age,educational-num,capital-gain,capital-loss,hours-per-week
count,45175.00,45175.00,45175.00,45175.00,45175.00
mean,0.00,0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00
min,-1.63,-3.57,-0.15,-0.22,-3.33
25%,-0.80,-0.44,-0.15,-0.22,-0.08
50%,-0.12,-0.05,-0.15,-0.22,-0.08
75%,0.64,1.13,-0.15,-0.22,0.34
max,3.89,2.30,13.17,10.53,4.84


## Resumen Final

In [ ]:
print(f"Dimensiones finales: {X.shape}")
print(f"\nTipos de datos:")
print(X.dtypes.value_counts())
print(f"\nNulos restantes: {X.isnull().sum().sum()}") #Ayuda de IA por unos fallos que tuve, se solucionó.
X.head()

Dimensiones finales: (45175, 32)

Tipos de datos:
bool       23
float64     5
int64       4
Name: count, dtype: int64

Nulos restantes: 0


,age,educational-num,marital-status,relationship,gender,capital-gain,capital-loss,hours-per-week,native-country,workclass_Local-gov,...,occupation_Priv-house-serv,occupation_Prof-specialty,occupation_Protective-serv,occupation_Sales,occupation_Tech-support,occupation_Transport-moving,race_Asian-Pac-Islander,race_Black,race_Other,race_White
0,-1.025801,-1.222440,4,3,1,-0.146811,-0.218899,-0.078493,1,False,...,False,False,False,False,False,False,False,True,False,False
1,-0.042086,-0.438652,2,0,1,-0.146811,-0.218899,0.754313,1,False,...,False,False,False,False,False,False,False,False,False,True
2,-0.798790,0.737029,2,0,1,-0.146811,-0.218899,-0.078493,1,True,...,False,False,True,False,False,False,False,False,False,True
3,0.411937,-0.046758,2,0,1,0.876868,-0.218899,-0.078493,1,False,...,False,False,False,False,False,False,False,True,False,False
5,-0.344767,-1.614334,4,1,1,-0.146811,-0.218899,-0.911299,1,False,...,False,False,False,False,False,False,False,False,False,True


## Conclusiones

Tras el proceso de selección y transformación, el dataset queda preparado para el modelado:

- Se han eliminado fnlwgt (sin valor para prediciiones) y education (redundante con educational-num).
- Se ha aplicado Label Encoding a variables binarias o de pocas categorías: gender, marital-status y relationship.
- Se ha aplicado One-Hot Encoding a variables nominales con más categorías: workclass, occupation y race.
- native-country se ha binarizado manualmente agrupando los países minoritarios para evitar dispersión.
- Las variables numéricas han sido escaladas con StandardScaler.
- El dataset final es completamente numérico y sin nulos, listo para Random Forest y LightGBM.

In [10]:
X.to_csv('data/X_processed.csv', index=False)
y.to_csv('data/y_processed.csv', index=False)

print("Datos guardados.")

Datos guardados.
